# HMM Categorical V1 — NVIDIA Developer Journey

**Goal:** Build the first categorical HMM using the weekly GMM cluster table.

This notebook uses:

- `dev_gmm_weekly_clusters_v1` as the weekly observed behavior sequence.
- `dev_lifecycle_cluster_membership_v11_final` as the static lifecycle/HDBSCAN interpretation layer.

Conceptually:

```text
Weekly GMM cluster = observed weekly behavior state
Categorical HMM = hidden journey-state transition model
V11 lifecycle/HDBSCAN cluster = static developer segment used for business interpretation
```

This notebook is designed to run locally or in Google Colab Pro/Pro+.

This variant is gap-aware and reproducible:
- keeps low-posterior weeks (no posterior-drop filtering),
- inserts explicit missing-week observations,
- uses deterministic hash-based sampling.


In [6]:
# ============================================================
# Optional Colab setup
# ============================================================

# If running in Colab, uncomment these lines:
# from google.colab import drive
# drive.mount('/content/drive')

# Then set PROJECT_DIR to wherever your DuckDB/parquet files live in Drive.
# Example:
# PROJECT_DIR = "/content/drive/MyDrive/NVIDIA Industry Project"

# If running locally, keep PROJECT_DIR as "."
PROJECT_DIR = "."

print("PROJECT_DIR:", PROJECT_DIR)


PROJECT_DIR: .


In [7]:
# ============================================================
# Install packages if needed
# ============================================================

# In Colab, upload requirements_hmm_categorical_v1.txt or place it in PROJECT_DIR,
# then uncomment one of the following:

# ============================================================
# Install packages if needed
# ============================================================

# In Colab, upload requirements_hmm_categorical_v1.txt or place it in PROJECT_DIR,
# then uncomment one of the following:

#!pip install -r "/content/drive/MyDrive/NVIDIA Industry Project/requirements_hmm_categorical_v1.txt"

# Or install directly:
# !pip install duckdb pandas numpy scikit-learn hmmlearn matplotlib pyarrow

# Or install directly:
# !pip install duckdb pandas numpy scikit-learn hmmlearn matplotlib pyarrow


In [8]:
# ============================================================
# Imports
# ============================================================

import os
from pathlib import Path
import warnings

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from hmmlearn.hmm import CategoricalHMM

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

print("Imports loaded.")


Imports loaded.


In [9]:
# ============================================================
# Paths and configurable parameters
# ============================================================

PROJECT_DIR = Path(PROJECT_DIR)

DB_PATH = PROJECT_DIR / "developer_project.duckdb"

# Folder containing exported parquet files, if you are loading tables from parquet.
# Update this path if needed.
PARQUET_DIR = PROJECT_DIR

GMM_WEEKLY_TABLE = "dev_gmm_weekly_clusters_v1"
V11_FINAL_TABLE = "dev_lifecycle_cluster_membership_v11_final"

# First-pass modeling choices
VALID_STRATA = ["active", "cooling", "at_risk"]  # use ["active", "at_risk"] if cooling is empty
MIN_WEEKS_PER_DEV = 6
MAX_DEVELOPERS = None  # None = use all eligible developers; set an int to cap for testing
MIN_GMM_POSTERIOR = 0.50  # retained as metadata threshold; weeks are not dropped

# HMM candidates
N_HIDDEN_STATE_OPTIONS = [2, 3, 4, 5]
SEED = 42

# Output directory for small experiment summaries
OUTPUT_DIR = PROJECT_DIR / "outputs" / "hmm_categorical_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DB_PATH:", DB_PATH)
print("PARQUET_DIR:", PARQUET_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


DB_PATH: developer_project.duckdb
PARQUET_DIR: .
OUTPUT_DIR: outputs/hmm_categorical_v2


In [10]:
# ============================================================
# Connect to DuckDB
# ============================================================

con = duckdb.connect(str(DB_PATH))
print("Connected to DuckDB:", DB_PATH)


Connected to DuckDB: developer_project.duckdb


## Load required tables from Parquet if needed

This cell is useful in Colab if your DuckDB file does not already contain the tables. It safely uses `CREATE OR REPLACE TABLE`.

Expected parquet filenames:

```text
dev_gmm_weekly_clusters_v1.parquet
dev_lifecycle_cluster_membership_v11_final.parquet
```

If your DuckDB already has these tables, this cell will simply skip loading unless you set `LOAD_FROM_PARQUET = True`.

In [11]:
# ============================================================
# Optional: Load required tables from parquet
# ============================================================

LOAD_FROM_PARQUET = False  # Change to True if running in Colab from parquet exports

required_parquets = {
    GMM_WEEKLY_TABLE: PARQUET_DIR / f"{GMM_WEEKLY_TABLE}.parquet",
    V11_FINAL_TABLE: PARQUET_DIR / f"{V11_FINAL_TABLE}.parquet",
}

if LOAD_FROM_PARQUET:
    for table_name, parquet_path in required_parquets.items():
        if parquet_path.exists():
            print(f"Loading {parquet_path.name} into {table_name}...")
            con.execute(f"""
                CREATE OR REPLACE TABLE {table_name} AS
                SELECT *
                FROM read_parquet('{parquet_path.as_posix()}')
            """)
            print(f"Loaded {table_name}.")
        else:
            raise FileNotFoundError(f"Missing parquet file: {parquet_path}")
else:
    print("LOAD_FROM_PARQUET=False. Using tables already present in DuckDB.")


LOAD_FROM_PARQUET=False. Using tables already present in DuckDB.


In [12]:
# ============================================================
# Validate required tables exist
# ============================================================

tables = con.execute("SHOW TABLES").df()
display(tables)

existing_tables = set(tables["name"].tolist())

for table in [GMM_WEEKLY_TABLE, V11_FINAL_TABLE]:
    if table not in existing_tables:
        raise ValueError(
            f"Required table not found: {table}. "
            "Either load it from parquet by setting LOAD_FROM_PARQUET=True, "
            "or make sure it already exists in developer_project.duckdb."
        )

print("Required tables found.")


,name
0,activity_base_v2
1,activity_dictionary_v2
2,activity_effort_mapping_ai_v2
3,activity_final
4,activity_labeled_v2
...,...
132,developer_clusters_v3_corr_groups_dormant
133,developer_clusters_v3_dormant
134,developer_universe_v2
135,sdk_download_final


Required tables found.


In [13]:
# ============================================================
# Inspect weekly GMM table
# ============================================================

gmm_summary = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT developer_id) AS n_developers,
    MIN(week_start) AS min_week,
    MAX(week_start) AS max_week,
    COUNT(DISTINCT gmm_weekly_cluster_id) AS n_gmm_clusters
FROM {GMM_WEEKLY_TABLE}
""").df()

display(gmm_summary)

gmm_dist = con.execute(f"""
SELECT
    gmm_weekly_cluster_id,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT developer_id) AS n_developers,
    AVG(gmm_weekly_max_posterior) AS avg_max_posterior
FROM {GMM_WEEKLY_TABLE}
GROUP BY gmm_weekly_cluster_id
ORDER BY gmm_weekly_cluster_id
""").df()

display(gmm_dist)


,n_rows,n_developers,min_week,max_week,n_gmm_clusters
0,14908480,7660278,2019-12-30,2026-03-09,3


,gmm_weekly_cluster_id,n_rows,n_developers,avg_max_posterior
0,0,8992453,4302668,0.999994
1,1,382890,177375,0.996337
2,2,5533137,4965049,0.999726


In [14]:
# ============================================================
# Inspect V11 lifecycle/HDBSCAN table
# ============================================================

v11_summary = con.execute(f"""
SELECT
    stratum,
    COUNT(*) AS n_developers,
    COUNT(DISTINCT cluster_key) AS n_cluster_keys,
    AVG(cluster_probability) AS avg_cluster_probability,
    AVG(outlier_score) AS avg_outlier_score
FROM {V11_FINAL_TABLE}
GROUP BY stratum
ORDER BY n_developers DESC
""").df()

display(v11_summary)


,stratum,n_developers,n_cluster_keys,avg_cluster_probability,avg_outlier_score
0,dormant,5304852,3,1.000000,0.000000
1,unactivated,1721230,1,1.000000,0.000000
2,at_risk,1580877,7,0.753555,0.124177
3,active,418049,7,0.655923,0.109012
4,cooling,356500,8,0.709506,0.107562


In [15]:
# ============================================================
# Check weekly GMM coverage by lifecycle stratum
# ============================================================

coverage_by_stratum = con.execute(f"""
SELECT
    c.stratum,
    COUNT(*) AS n_weekly_rows,
    COUNT(DISTINCT g.developer_id) AS n_developers,
    COUNT(DISTINCT g.gmm_weekly_cluster_id) AS n_gmm_clusters,
    AVG(g.gmm_weekly_max_posterior) AS avg_gmm_posterior
FROM {GMM_WEEKLY_TABLE} g
LEFT JOIN {V11_FINAL_TABLE} c
    ON g.developer_id = c.developer_id
GROUP BY c.stratum
ORDER BY n_developers DESC
""").df()

display(coverage_by_stratum)


,stratum,n_weekly_rows,n_developers,n_gmm_clusters,avg_gmm_posterior
0,dormant,9129327,5304852,3,0.999875
1,at_risk,3458604,1580877,3,0.999791
2,active,1395180,418049,3,0.999421
3,cooling,925369,356500,3,0.999679


In [16]:
# ============================================================
# Create HMM input temp table
# ============================================================

valid_strata_sql = ", ".join([f"'{s}'" for s in VALID_STRATA])

con.execute(f"""
CREATE OR REPLACE TEMP TABLE hmm_gmm_input_temp AS
SELECT
    g.developer_id,
    CAST(g.week_start AS DATE) AS week_start,
    CAST(g.gmm_weekly_cluster_id AS INTEGER) AS gmm_weekly_cluster_id,
    CAST(g.gmm_weekly_max_posterior AS DOUBLE) AS gmm_weekly_max_posterior,
    c.stratum,
    c.cluster_key,
    c.cluster_probability,
    c.outlier_score,
    c.adoption_direction
FROM {GMM_WEEKLY_TABLE} g
JOIN {V11_FINAL_TABLE} c
    ON g.developer_id = c.developer_id
WHERE c.stratum IN ({valid_strata_sql})
""")

input_summary = con.execute("""
SELECT
    stratum,
    COUNT(*) AS n_weekly_rows,
    COUNT(DISTINCT developer_id) AS n_developers,
    COUNT(DISTINCT gmm_weekly_cluster_id) AS n_gmm_clusters,
    AVG(gmm_weekly_max_posterior) AS avg_gmm_posterior
FROM hmm_gmm_input_temp
GROUP BY stratum
ORDER BY n_developers DESC
""").df()

display(input_summary)


,stratum,n_weekly_rows,n_developers,n_gmm_clusters,avg_gmm_posterior
0,at_risk,3458604,1580877,3,0.999791
1,active,1395180,418049,3,0.999421
2,cooling,925369,356500,3,0.999679


In [17]:
# ============================================================
# Select developers with enough weekly observations
# Full-data mode: MAX_DEVELOPERS=None means no LIMIT is applied.
# ============================================================

limit_sql = "" if MAX_DEVELOPERS is None else f"LIMIT {int(MAX_DEVELOPERS)}"
order_sql = "developer_id" if MAX_DEVELOPERS is None else f"hash(CAST(developer_id AS VARCHAR) || '{SEED}')"

eligible_devs = con.execute(f"""
WITH dev_counts AS (
    SELECT
        developer_id,
        COUNT(*) AS n_weeks,
        AVG(gmm_weekly_max_posterior) AS avg_gmm_confidence
    FROM hmm_gmm_input_temp
    GROUP BY developer_id
    HAVING COUNT(*) >= {MIN_WEEKS_PER_DEV}
)
SELECT developer_id, n_weeks, avg_gmm_confidence
FROM dev_counts
ORDER BY {order_sql}
{limit_sql}
""").df()

print("Eligible developers:", eligible_devs.shape[0])
display(eligible_devs.describe(include="all"))

con.register("eligible_devs_sample", eligible_devs)


Eligible developers: 176965


,developer_id,n_weeks,avg_gmm_confidence
count,176965,176965.000000,176965.000000
unique,176965,NaN,NaN
top,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,NaN,NaN
freq,1,NaN,NaN
mean,NaN,14.369542,0.999724
std,NaN,16.114001,0.002899
min,NaN,6.000000,0.808669
25%,NaN,7.000000,0.999998
50%,NaN,9.000000,0.999999
75%,NaN,15.000000,0.999999


In [18]:
# ============================================================
# Load sampled sequence data into pandas
# ============================================================

hmm_df = con.execute("""
SELECT h.*
FROM hmm_gmm_input_temp h
JOIN eligible_devs_sample e
    ON h.developer_id = e.developer_id
ORDER BY h.developer_id, h.week_start
""").df()

print("hmm_df shape (observed weeks):", hmm_df.shape)
display(hmm_df.head())

# Explicitly represent missing weeks as a dedicated observation category
# so long gaps are not treated as one-step jumps.
hmm_df["week_start"] = pd.to_datetime(hmm_df["week_start"])

expanded_parts = []
for dev_id, g in hmm_df.groupby("developer_id", sort=False):
    g = g.sort_values("week_start").copy()
    full_weeks = pd.date_range(
        start=g["week_start"].min(),
        end=g["week_start"].max(),
        freq="W-MON",
    )
    frame = pd.DataFrame({"week_start": full_weeks})
    frame["developer_id"] = dev_id
    merged = frame.merge(g, on=["developer_id", "week_start"], how="left")

    # Backfill static interpretation columns per developer
    for col in ["stratum", "cluster_key", "cluster_probability", "outlier_score", "adoption_direction"]:
        if col in merged.columns:
            merged[col] = merged[col].ffill().bfill()

    merged["missing_week_flag"] = merged["gmm_weekly_cluster_id"].isna().astype(int)
    merged["gmm_weekly_max_posterior"] = merged["gmm_weekly_max_posterior"].fillna(0.0)
    # Dedicated observed category for missing/no-activity weeks
    merged["gmm_weekly_cluster_id"] = merged["gmm_weekly_cluster_id"].fillna(-1).astype(int)

    expanded_parts.append(merged)

hmm_df = pd.concat(expanded_parts, ignore_index=True).sort_values(["developer_id", "week_start"]).reset_index(drop=True)

print("hmm_df shape (gap-filled weeks):", hmm_df.shape)
print("Missing-week rows added:", int(hmm_df["missing_week_flag"].sum()))
display(hmm_df.head())


hmm_df shape (observed weeks): (2542906, 9)


,developer_id,week_start,gmm_weekly_cluster_id,gmm_weekly_max_posterior,stratum,cluster_key,cluster_probability,outlier_score,adoption_direction
0,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2021-11-08,2,1.000000,at_risk,at_risk_5,1.0,0.0,at_risk
1,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2022-02-21,0,0.999999,at_risk,at_risk_5,1.0,0.0,at_risk
2,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2022-08-15,0,0.999996,at_risk,at_risk_5,1.0,0.0,at_risk
3,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2022-08-29,0,0.999996,at_risk,at_risk_5,1.0,0.0,at_risk
4,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2023-05-01,0,0.999996,at_risk,at_risk_5,1.0,0.0,at_risk


hmm_df shape (gap-filled weeks): (26093521, 10)
Missing-week rows added: 23550615


,week_start,developer_id,gmm_weekly_cluster_id,gmm_weekly_max_posterior,stratum,cluster_key,cluster_probability,outlier_score,adoption_direction,missing_week_flag
0,2021-11-08,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2,1.0,at_risk,at_risk_5,1.0,0.0,at_risk,0
1,2021-11-15,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,-1,0.0,at_risk,at_risk_5,1.0,0.0,at_risk,1
2,2021-11-22,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,-1,0.0,at_risk,at_risk_5,1.0,0.0,at_risk,1
3,2021-11-29,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,-1,0.0,at_risk,at_risk_5,1.0,0.0,at_risk,1
4,2021-12-06,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,-1,0.0,at_risk,at_risk_5,1.0,0.0,at_risk,1


In [19]:
# ============================================================
# Remap GMM cluster IDs to contiguous integer observations
# CategoricalHMM requires observations to be integer categories.
# ============================================================

unique_clusters = sorted(hmm_df["gmm_weekly_cluster_id"].dropna().unique().tolist())

cluster_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_clusters)}
reverse_cluster_id_map = {new_id: old_id for old_id, new_id in cluster_id_map.items()}

hmm_df["gmm_obs_id"] = hmm_df["gmm_weekly_cluster_id"].map(cluster_id_map).astype(int)

print("Original GMM cluster IDs:", unique_clusters)
print("Observation ID map:", cluster_id_map)
print("Number of observed GMM categories:", hmm_df["gmm_obs_id"].nunique())

display(
    hmm_df.groupby(["gmm_weekly_cluster_id", "gmm_obs_id"])
    .size()
    .reset_index(name="n_rows")
    .sort_values("gmm_obs_id")
)


Original GMM cluster IDs: [-1, 0, 1, 2]
Observation ID map: {-1: 0, 0: 1, 1: 2, 2: 3}
Number of observed GMM categories: 4


,gmm_weekly_cluster_id,gmm_obs_id,n_rows
0,-1,0,23550615
1,0,1,2087961
2,1,2,152626
3,2,3,302319


In [20]:
# ============================================================
# Build HMM X array and sequence lengths
# ============================================================

hmm_df = hmm_df.sort_values(["developer_id", "week_start"]).reset_index(drop=True)

lengths = hmm_df.groupby("developer_id").size().tolist()
X = hmm_df[["gmm_obs_id"]].astype(int).values

n_observed_gmm_clusters = hmm_df["gmm_obs_id"].nunique()

print("X shape:", X.shape)
print("Number of developer sequences:", len(lengths))
print("Total sequence length:", sum(lengths))
print("Average sequence length:", np.mean(lengths))
print("Median sequence length:", np.median(lengths))
print("Observed GMM categories:", n_observed_gmm_clusters)

assert sum(lengths) == X.shape[0]


X shape: (26093521, 1)
Number of developer sequences: 176965
Total sequence length: 26093521
Average sequence length: 147.45017941400843
Median sequence length: 144.0
Observed GMM categories: 4


In [21]:
# ============================================================
# Fit Categorical HMM models
# ============================================================

results = []
models = {}

for n_states in N_HIDDEN_STATE_OPTIONS:
    print(f"\nFitting CategoricalHMM with {n_states} hidden states...")

    model = CategoricalHMM(
        n_components=n_states,
        n_features=n_observed_gmm_clusters,
        n_iter=100,
        tol=1e-4,
        random_state=42,
        verbose=False
    )

    model.fit(X, lengths)
    log_likelihood = model.score(X, lengths)

    # Approximate parameter count for AIC/BIC:
    # start probs: n_states - 1
    # transition rows: n_states * (n_states - 1)
    # emission rows: n_states * (n_observed_categories - 1)
    n_params = (
        (n_states - 1)
        + n_states * (n_states - 1)
        + n_states * (n_observed_gmm_clusters - 1)
    )

    n_obs = X.shape[0]
    aic = -2 * log_likelihood + 2 * n_params
    bic = -2 * log_likelihood + n_params * np.log(n_obs)

    results.append({
        "n_hidden_states": n_states,
        "log_likelihood": log_likelihood,
        "aic": aic,
        "bic": bic,
        "n_obs": n_obs,
        "n_sequences": len(lengths),
        "n_observed_gmm_clusters": n_observed_gmm_clusters,
        "converged": model.monitor_.converged,
        "n_iter": model.monitor_.iter
    })

    models[n_states] = model

results_df = pd.DataFrame(results).sort_values("bic")
display(results_df)



Fitting CategoricalHMM with 2 hidden states...

Fitting CategoricalHMM with 3 hidden states...

Fitting CategoricalHMM with 4 hidden states...

Fitting CategoricalHMM with 5 hidden states...


,n_hidden_states,log_likelihood,aic,bic,n_obs,n_sequences,n_observed_gmm_clusters,converged,n_iter
1,3,-7.705262e+06,1.541056e+07,1.541081e+07,26093521,176965,4,True,100
3,5,-7.718364e+06,1.543681e+07,1.543739e+07,26093521,176965,4,True,100
2,4,-7.732651e+06,1.546536e+07,1.546576e+07,26093521,176965,4,True,100
0,2,-7.927003e+06,1.585402e+07,1.585416e+07,26093521,176965,4,True,75


In [22]:
# ============================================================
# Choose best model
# ============================================================

# Default: choose lowest BIC.
# You can override this manually after looking at interpretability.
BEST_N_STATES = int(results_df.iloc[0]["n_hidden_states"])

# Optional override:
# BEST_N_STATES = 4

best_model = models[BEST_N_STATES]

print("Selected hidden states:", BEST_N_STATES)


Selected hidden states: 3


In [23]:
# ============================================================
# Predict hidden journey states
# ============================================================

hmm_df["hmm_state"] = best_model.predict(X, lengths)

display(
    hmm_df[[
        "developer_id",
        "week_start",
        "stratum",
        "cluster_key",
        "gmm_weekly_cluster_id",
        "gmm_obs_id",
        "hmm_state"
    ]].head(20)
)


,developer_id,week_start,stratum,cluster_key,gmm_weekly_cluster_id,gmm_obs_id,hmm_state
0,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2021-11-08,at_risk,at_risk_5,2,3,1
1,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2021-11-15,at_risk,at_risk_5,-1,0,2
2,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2021-11-22,at_risk,at_risk_5,-1,0,2
3,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2021-11-29,at_risk,at_risk_5,-1,0,2
4,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2021-12-06,at_risk,at_risk_5,-1,0,2
5,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2021-12-13,at_risk,at_risk_5,-1,0,2
6,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2021-12-20,at_risk,at_risk_5,-1,0,2
7,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2021-12-27,at_risk,at_risk_5,-1,0,2
8,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2022-01-03,at_risk,at_risk_5,-1,0,2
9,02af006fa5c595b7a2f01b92d8a42cce12d589344f88e4...,2022-01-10,at_risk,at_risk_5,-1,0,2


## Interpret hidden states

For a categorical HMM, the emission matrix is one of the most important interpretation tools.

It answers:

```text
When the model is in hidden HMM state X, what weekly GMM cluster is it likely to emit?
```

Use this to give business-friendly names to HMM states after looking at what GMM clusters 0/1/2 mean.

In [24]:
# ============================================================
# Emission probabilities
# ============================================================

emission_probs = pd.DataFrame(
    best_model.emissionprob_,
    index=[f"hmm_state_{i}" for i in range(BEST_N_STATES)],
    columns=[f"gmm_obs_{i}_orig_{reverse_cluster_id_map[i]}" for i in range(n_observed_gmm_clusters)]
)

display(emission_probs)

# Most likely GMM emission for each HMM state
dominant_emissions = (
    emission_probs
    .idxmax(axis=1)
    .reset_index()
    .rename(columns={"index": "hmm_state", 0: "dominant_emission"})
)

display(dominant_emissions)


,gmm_obs_0_orig_-1,gmm_obs_1_orig_0,gmm_obs_2_orig_1,gmm_obs_3_orig_2
hmm_state_0,0.397138,0.541265,0.061192,0.000405
hmm_state_1,0.002141,0.556119,0.052593,0.389147
hmm_state_2,0.966345,0.031071,0.000255,0.002330


,hmm_state,dominant_emission
0,hmm_state_0,gmm_obs_1_orig_0
1,hmm_state_1,gmm_obs_1_orig_0
2,hmm_state_2,gmm_obs_0_orig_-1


In [25]:
# ============================================================
# Transition matrix
# ============================================================

transition_matrix = pd.DataFrame(
    best_model.transmat_,
    index=[f"from_hmm_state_{i}" for i in range(BEST_N_STATES)],
    columns=[f"to_hmm_state_{i}" for i in range(BEST_N_STATES)]
)

display(transition_matrix)

transition_long = (
    transition_matrix
    .reset_index()
    .melt(id_vars="index", var_name="to_state", value_name="transition_probability")
    .rename(columns={"index": "from_state"})
)

display(transition_long.sort_values("transition_probability", ascending=False).head(20))


,to_hmm_state_0,to_hmm_state_1,to_hmm_state_2
from_hmm_state_0,0.866774,0.114366,0.018860
from_hmm_state_1,0.274328,0.189655,0.536017
from_hmm_state_2,0.006058,0.005856,0.988086


,from_state,to_state,transition_probability
8,from_hmm_state_2,to_hmm_state_2,0.988086
0,from_hmm_state_0,to_hmm_state_0,0.866774
7,from_hmm_state_1,to_hmm_state_2,0.536017
1,from_hmm_state_1,to_hmm_state_0,0.274328
4,from_hmm_state_1,to_hmm_state_1,0.189655
3,from_hmm_state_0,to_hmm_state_1,0.114366
6,from_hmm_state_0,to_hmm_state_2,0.018860
2,from_hmm_state_2,to_hmm_state_0,0.006058
5,from_hmm_state_2,to_hmm_state_1,0.005856


In [26]:
# ============================================================
# State profiles
# ============================================================

state_profiles = (
    hmm_df
    .groupby("hmm_state")
    .agg(
        n_weekly_rows=("developer_id", "size"),
        n_developers=("developer_id", "nunique"),
        avg_gmm_posterior=("gmm_weekly_max_posterior", "mean"),
        most_common_gmm_cluster=("gmm_weekly_cluster_id", lambda x: x.value_counts().index[0]),
        stratum_mode=("stratum", lambda x: x.value_counts().index[0])
    )
    .reset_index()
)

state_profiles["share_of_rows"] = state_profiles["n_weekly_rows"] / state_profiles["n_weekly_rows"].sum()

display(state_profiles)


,hmm_state,n_weekly_rows,n_developers,avg_gmm_posterior,most_common_gmm_cluster,stratum_mode,share_of_rows
0,0,1837545,120268,0.641259,0,active,0.070422
1,1,441393,176965,0.998701,2,at_risk,0.016916
2,2,23814583,148065,0.038750,-1,at_risk,0.912663


In [27]:
# ============================================================
# HMM state distribution by lifecycle stratum
# ============================================================

state_by_stratum = (
    hmm_df
    .groupby(["stratum", "hmm_state"])
    .size()
    .reset_index(name="n_rows")
)

state_by_stratum["share_within_stratum"] = (
    state_by_stratum["n_rows"]
    / state_by_stratum.groupby("stratum")["n_rows"].transform("sum")
)

display(state_by_stratum.sort_values(["stratum", "hmm_state"]))


,stratum,hmm_state,n_rows,share_within_stratum
0,active,0,795480,0.172593
1,active,1,110519,0.023979
2,active,2,3702988,0.803428
3,at_risk,0,727976,0.044210
4,at_risk,1,249548,0.015155
5,at_risk,2,15488747,0.940635
6,cooling,0,314089,0.062589
7,cooling,1,81326,0.016206
8,cooling,2,4622848,0.921205


In [28]:
# ============================================================
# HMM state distribution by V11/HDBSCAN cluster
# ============================================================

state_by_v11_cluster = (
    hmm_df
    .groupby(["stratum", "cluster_key", "hmm_state"])
    .size()
    .reset_index(name="n_rows")
)

state_by_v11_cluster["share_within_cluster"] = (
    state_by_v11_cluster["n_rows"]
    / state_by_v11_cluster.groupby(["stratum", "cluster_key"])["n_rows"].transform("sum")
)

display(
    state_by_v11_cluster
    .sort_values(["stratum", "cluster_key", "share_within_cluster"], ascending=[True, True, False])
    .head(100)
)


,stratum,cluster_key,hmm_state,n_rows,share_within_cluster
2,active,active_0,2,50,0.909091
0,active,active_0,0,3,0.054545
1,active,active_0,1,2,0.036364
5,active,active_1,2,3828,0.867044
3,active,active_1,0,400,0.090600
4,active,active_1,1,187,0.042356
6,active,active_3,0,136616,0.774296
8,active,active_3,2,26330,0.149230
7,active,active_3,1,13493,0.076474
10,active,active_5,2,296,0.996633


## Optional risk-oriented analysis

Because we do not automatically know which HMM state is “risky,” this section lets you manually define risky states after inspecting the emission probabilities and state profiles.

Example:
- If `hmm_state_0` mostly emits a low/no-activity GMM cluster, it may be a risky or inactive state.
- If `hmm_state_2` mostly emits high-intent weekly behavior, it may be a healthy state.

Update `RISKY_HMM_STATES` after interpretation.

In [29]:
# ============================================================
# Manual risky state assignment
# ============================================================

# Update after inspecting emission_probs and state_profiles.
RISKY_HMM_STATES = []  # example: [0]

hmm_df["is_risky_hmm_state"] = hmm_df["hmm_state"].isin(RISKY_HMM_STATES).astype(int)

if RISKY_HMM_STATES:
    risk_by_cluster = (
        hmm_df
        .groupby(["stratum", "cluster_key"])
        .agg(
            n_weekly_rows=("developer_id", "size"),
            n_developers=("developer_id", "nunique"),
            risk_state_share=("is_risky_hmm_state", "mean")
        )
        .reset_index()
        .sort_values("risk_state_share", ascending=False)
    )

    display(risk_by_cluster.head(50))
else:
    print("RISKY_HMM_STATES is empty. Inspect emission_probs/state_profiles first, then fill this in.")


RISKY_HMM_STATES is empty. Inspect emission_probs/state_profiles first, then fill this in.


In [30]:
# ============================================================
# Save small experiment outputs locally
# We are NOT saving model assignments to DuckDB yet because this is exploratory.
# ============================================================

results_df.to_csv(OUTPUT_DIR / "hmm_categorical_model_comparison.csv", index=False)
state_profiles.to_csv(OUTPUT_DIR / "hmm_categorical_state_profiles.csv", index=False)
transition_matrix.to_csv(OUTPUT_DIR / "hmm_categorical_transition_matrix.csv")
transition_long.to_csv(OUTPUT_DIR / "hmm_categorical_transition_matrix_long.csv", index=False)
emission_probs.to_csv(OUTPUT_DIR / "hmm_categorical_emission_probabilities.csv")
state_by_stratum.to_csv(OUTPUT_DIR / "hmm_categorical_state_by_stratum.csv", index=False)
state_by_v11_cluster.to_csv(OUTPUT_DIR / "hmm_categorical_state_by_v11_cluster.csv", index=False)

print("Saved summary outputs to:", OUTPUT_DIR)


Saved summary outputs to: outputs/hmm_categorical_v2


In [31]:
con.close()

# Interpretation notes to fill in after running

After running this notebook, answer:

1. What does each weekly GMM cluster mean behaviorally?
2. What does each HMM hidden state emit most often?
3. Which HMM states look healthy, transitional, or risky?
4. Which V11/HDBSCAN clusters spend the most time in risky HMM states?
5. Which clusters show the strongest opportunity for personalized outreach?

Suggested final business framing:

```text
The weekly GMM model gives each developer-week an observed behavior state. The categorical HMM uses these observed weekly states to learn hidden journey states and transition probabilities over time. By joining the HMM states back to the V11 lifecycle/HDBSCAN clusters, we can identify which developer segments are following healthy adoption paths versus which segments are trending toward lower engagement or at-risk behavior.
```